 ## Email Data Analysis
Email analysis involves extracting insights from email datasets. It includes assessing communication patterns, such as send/receive frequencies. Content analysis using natural language processing reveals keywords, and topics. Visualization techniques represent trends, network structures, and other patterns for comprehensive understanding

In [1]:
import numpy as np
import pandas  as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

## Converting Gmail Emails to CSV

We have a Gmail dataset in mbox format. This project converts the data into a CSV file for further analysis.

In [2]:
# Loading the mbox file

import mailbox
mbox = mailbox.mbox("..\\All mail Including Spam and Trash.mbox")
mbox

In [3]:
## Displaying the keys of the first email in the mbox to understand the structure of the data

keys2 = list(mbox[0].keys())

if 'From' in mbox[0]:
    print("Key 'From' found in the email keys.")
else:
    print("Key 'From' not found in the email keys.")

Key 'From' found in the email keys.


In [4]:
import csv

with open('../data/email_data.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['From', 'To', 'Subject', 'Date', 'Thread', 'Labels'])
    for message in mbox:
        writer.writerow([message.get('From', ''),
                         message.get('To', ''),
                         message.get('Subject', ''),
                         message.get('Date', ''),
                         message.get('X-GM-THRID', ''),
                         message.get('X-Gmail-Labels', '')])

In [5]:
df = pd.read_csv('../data/email_data.csv')
df.head()

,From,To,Subject,Date,Thread,Labels
0,=?utf-8?q?Indro_Vicari_via_beBee?= <alert@noti...,bc090402541@gmail.com,=?utf-8?q?Nuova_posizione_di_Programmatore_Fro...,"Mon, 13 Nov 2023 17:16:53 +0000",1782469952194646866,"Inbox,Category Updates,Unread"
1,"""Jason @ ML Mastery"" <jason@MachineLearningMas...",bc090402541@gmail.com,Looking To Stay Current in Data Analytics?,"Mon, 13 Nov 2023 15:00:00 +0000 (UTC)",1782461343277158936,"Inbox,Category Updates,Unread"
2,=?utf-8?B?bGF2b3JvIHRyYWJham8ub3Jn?= <info@tra...,"""bc090402541@gmail.com"" <bc090402541@gmail.com>",Offerte di lavoro per SQL Developer a Rome,"Tue, 14 Nov 2023 12:33:11 +0100",1782538925643770298,"Inbox,Category Updates,Unread"
3,=?utf-8?B?bGF2b3JvIHRyYWJham8ub3Jn?= <info@tra...,"""bc090402541@gmail.com"" <bc090402541@gmail.com>",Offerte di lavoro per Junior Java Developer a ...,"Wed, 15 Nov 2023 12:33:24 +0100",1782629536219841707,"Inbox,Category Updates,Unread"
4,"""OpenCV.org"" <newsletter@opencv.org>",uzair shafique <bc090402541@gmail.com>,"Donate To Keep OpenCV Free To All, Impact Summ...","Wed, 15 Nov 2023 20:38:11 +0000 (UTC)",1782663811768277342,"Inbox,Category Updates,Unread"


In [6]:
# Converting the 'Date' column to datetime format, handling errors by coercing invalid formats to NaT
df["Date"] = df["Date"].apply(lambda x: pd.to_datetime(x, errors='coerce'))

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14734 entries, 0 to 14733
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   From     14734 non-null  str   
 1   To       14721 non-null  str   
 2   Subject  14687 non-null  str   
 3   Date     14734 non-null  object
 4   Thread   14734 non-null  int64 
 5   Labels   14734 non-null  str   
dtypes: int64(1), object(1), str(4)
memory usage: 690.8+ KB


In [8]:
import re

# Function to extract email addresses from a string
def extract_email(email_str):
    if pd.isna(email_str):
        return None
    match = re.findall(r'[\w\.-]+@[\w\.-]+', email_str)
    return match[0] if match else None

df['From'] = df['From'].apply(extract_email)
df['To'] = df['To'].apply(extract_email)
df.head()

,From,To,Subject,Date,Thread,Labels
0,alert@notification.bebee.com,bc090402541@gmail.com,=?utf-8?q?Nuova_posizione_di_Programmatore_Fro...,2023-11-13 17:16:53+00:00,1782469952194646866,"Inbox,Category Updates,Unread"
1,jason@MachineLearningMastery.com,bc090402541@gmail.com,Looking To Stay Current in Data Analytics?,2023-11-13 15:00:00+00:00,1782461343277158936,"Inbox,Category Updates,Unread"
2,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per SQL Developer a Rome,2023-11-14 12:33:11+01:00,1782538925643770298,"Inbox,Category Updates,Unread"
3,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per Junior Java Developer a ...,2023-11-15 12:33:24+01:00,1782629536219841707,"Inbox,Category Updates,Unread"
4,newsletter@opencv.org,bc090402541@gmail.com,"Donate To Keep OpenCV Free To All, Impact Summ...",2023-11-15 20:38:11+00:00,1782663811768277342,"Inbox,Category Updates,Unread"


In [9]:

from email.header import decode_header

# Function to decode email subjects that may be encoded in various formats
def clean_subject(subject):
    if pd.isna(subject):
        return None

    decoded_parts = decode_header(subject)
    cleaned_subject = ''
    for part, encoding in decoded_parts:
        if isinstance(part, bytes):
            cleaned_subject += part.decode(encoding or 'utf-8', errors='ignore')
        else:
            cleaned_subject += part
    return cleaned_subject


In [10]:
# Function to further clean the subject by removing common prefixes and extra whitespace
def clean_text(text):
    if pd.isna(text):
        return None

    text = re.sub(r'^(Re:|Fwd:)\s*', '', text, flags=re.IGNORECASE)
    text = text.replace('_',' ')
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df['Subject'] = df['Subject'].apply(clean_subject)
df['Subject'] = df['Subject'].apply(clean_text)
df


,From,To,Subject,Date,Thread,Labels
0,alert@notification.bebee.com,bc090402541@gmail.com,Nuova posizione di Programmatore Front End in ...,2023-11-13 17:16:53+00:00,1782469952194646866,"Inbox,Category Updates,Unread"
1,jason@MachineLearningMastery.com,bc090402541@gmail.com,Looking To Stay Current in Data Analytics?,2023-11-13 15:00:00+00:00,1782461343277158936,"Inbox,Category Updates,Unread"
2,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per SQL Developer a Rome,2023-11-14 12:33:11+01:00,1782538925643770298,"Inbox,Category Updates,Unread"
3,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per Junior Java Developer a ...,2023-11-15 12:33:24+01:00,1782629536219841707,"Inbox,Category Updates,Unread"
4,newsletter@opencv.org,bc090402541@gmail.com,"Donate To Keep OpenCV Free To All, Impact Summ...",2023-11-15 20:38:11+00:00,1782663811768277342,"Inbox,Category Updates,Unread"
...,...,...,...,...,...,...
14729,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per SQL Developer in Rome,2023-10-19 11:41:05+02:00,1780176352685898505,"Inbox,Category Updates,Unread"
14730,contact@info.foodpanda.pk,bc090402541@gmail.com,Khaas deals you wouldn't wanna miss 👉🏻,2023-08-13 12:19:49+00:00,1774119953401055792,"Inbox,Category Updates,Unread"
14731,alert@notification.bebee.com,bc090402541@gmail.com,Nuova posizione di Database / SQL Developer in...,2023-07-21 10:09:09+00:00,1772024390878438311,"Inbox,Category Updates,Unread"
14732,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per Web Developer in Salerno,2023-08-06 11:56:44+02:00,1773473160922129739,"Inbox,Category Updates,Unread"


In [11]:

# Labeling emails as 'Sent' or 'Received' based on the 'From' field

my_email = "bc09402541@gmail.com"
def label_email(row):
    if row['From'] == my_email:
        return 'Sent'
    else:
        return 'Received'

df['Email_Type'] = df.apply(label_email, axis=1)
df


,From,To,Subject,Date,Thread,Labels,Email_Type
0,alert@notification.bebee.com,bc090402541@gmail.com,Nuova posizione di Programmatore Front End in ...,2023-11-13 17:16:53+00:00,1782469952194646866,"Inbox,Category Updates,Unread",Received
1,jason@MachineLearningMastery.com,bc090402541@gmail.com,Looking To Stay Current in Data Analytics?,2023-11-13 15:00:00+00:00,1782461343277158936,"Inbox,Category Updates,Unread",Received
2,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per SQL Developer a Rome,2023-11-14 12:33:11+01:00,1782538925643770298,"Inbox,Category Updates,Unread",Received
3,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per Junior Java Developer a ...,2023-11-15 12:33:24+01:00,1782629536219841707,"Inbox,Category Updates,Unread",Received
4,newsletter@opencv.org,bc090402541@gmail.com,"Donate To Keep OpenCV Free To All, Impact Summ...",2023-11-15 20:38:11+00:00,1782663811768277342,"Inbox,Category Updates,Unread",Received
...,...,...,...,...,...,...,...
14729,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per SQL Developer in Rome,2023-10-19 11:41:05+02:00,1780176352685898505,"Inbox,Category Updates,Unread",Received
14730,contact@info.foodpanda.pk,bc090402541@gmail.com,Khaas deals you wouldn't wanna miss 👉🏻,2023-08-13 12:19:49+00:00,1774119953401055792,"Inbox,Category Updates,Unread",Received
14731,alert@notification.bebee.com,bc090402541@gmail.com,Nuova posizione di Database / SQL Developer in...,2023-07-21 10:09:09+00:00,1772024390878438311,"Inbox,Category Updates,Unread",Received
14732,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per Web Developer in Salerno,2023-08-06 11:56:44+02:00,1773473160922129739,"Inbox,Category Updates,Unread",Received


In [12]:
# Extracting the main label from the 'Labels' column to categorize emails into broader categories
def extract_main_label(label):
    if pd.isna(label):
        return "Unknown"

    if "Promotions" in label:
        return "Promotions"
    elif "Social" in label:
        return "Social"
    elif "Updates" in label:
        return "Updates"
    elif "Personal" in label:
        return "Personal"
    elif "Important" in label:
        return "Important"
    else:
        return "Other"

df['Main_Label'] = df['Labels'].apply(extract_main_label)
df[df['Main_Label'].str.contains("Important")].head()

,From,To,Subject,Date,Thread,Labels,Email_Type,Main_Label
2076,bc090402541@gmail.com,bc090402541@gmail.com,NaN,2023-07-10 09:55:38+02:00,1771019403880573430,"Sent,Inbox,Important,Opened",Received,Important
2387,bc090402541@gmail.com,bc090402541@gmail.com,NaN,2023-07-10 14:15:09+02:00,1771035738767687040,"Sent,Inbox,Important,Opened",Received,Important
6556,bc090402541@gmail.com,bc090402541@gmail.com,NaN,2022-12-27 03:27:52+01:00,1753332269670557175,"Sent,Inbox,Important,Opened",Received,Important
7043,bc090402541@gmail.com,bc090402541@gmail.com,I am sharing 'Uzair Shafique (1)' with you,2022-11-03 18:28:33+01:00,1748496794610763834,"Sent,Inbox,Important,Opened",Received,Important
8389,bc090402541@gmail.com,bc090402541@gmail.com,NaN,2022-10-13 10:24:21-07:00,1746594012769979922,"Sent,Inbox,Important,Opened",Received,Important


In [13]:
# Dropping the 'Labels' and 'Thread' columns as they are no longer needed for analysis
df.drop(columns=['Labels', 'Thread'], inplace=True)

In [14]:
df

,From,To,Subject,Date,Email_Type,Main_Label
0,alert@notification.bebee.com,bc090402541@gmail.com,Nuova posizione di Programmatore Front End in ...,2023-11-13 17:16:53+00:00,Received,Updates
1,jason@MachineLearningMastery.com,bc090402541@gmail.com,Looking To Stay Current in Data Analytics?,2023-11-13 15:00:00+00:00,Received,Updates
2,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per SQL Developer a Rome,2023-11-14 12:33:11+01:00,Received,Updates
3,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per Junior Java Developer a ...,2023-11-15 12:33:24+01:00,Received,Updates
4,newsletter@opencv.org,bc090402541@gmail.com,"Donate To Keep OpenCV Free To All, Impact Summ...",2023-11-15 20:38:11+00:00,Received,Updates
...,...,...,...,...,...,...
14729,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per SQL Developer in Rome,2023-10-19 11:41:05+02:00,Received,Updates
14730,contact@info.foodpanda.pk,bc090402541@gmail.com,Khaas deals you wouldn't wanna miss 👉🏻,2023-08-13 12:19:49+00:00,Received,Updates
14731,alert@notification.bebee.com,bc090402541@gmail.com,Nuova posizione di Database / SQL Developer in...,2023-07-21 10:09:09+00:00,Received,Updates
14732,info@trabajo.org,bc090402541@gmail.com,Offerte di lavoro per Web Developer in Salerno,2023-08-06 11:56:44+02:00,Received,Updates
